# Model Comparison: Multiple Approaches

This notebook trains and compares models using the fixed outputs from `02_preprocessing.ipynb`.

**Main purpose:** benchmark candidate models and choose the best-performing approach for monetary-relief prediction.

Models compared:
1. **Baseline**: Logistic Regression, Naive Bayes
2. **KNN**: k-Nearest Neighbors (on TF-IDF)
3. **Tree Ensemble**: Random Forest
4. **Neural Network**: Multi-layer Perceptron (ANN)
5. **Meta-Learner**: Voting ensemble over trained models

Evaluation: F1, Precision, Recall, PR-AUC, ROC-AUC

**Boundary:** no target redefinition or EDA logic here; this notebook consumes prepared train/test features only.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load train/test data from preprocessing
train_data = pd.read_csv('../data/processed/train_features.csv')
test_data = pd.read_csv('../data/processed/test_features.csv')

y_train = train_data['target']
y_test = test_data['target']
X_train = train_data.drop('target', axis=1)
X_test = test_data.drop('target', axis=1)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Target distribution (train): {y_train.value_counts().to_dict()}')

In [ ]:
# Prepare text features (TF-IDF for text-based models)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler

# Extract narrative column
narratives_train = X_train['narrative_clean'].fillna('')
narratives_test = X_test['narrative_clean'].fillna('')

# Fit TF-IDF on training narratives
tfidf = TfidfVectorizer(max_features=200, max_df=0.8, min_df=2, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(narratives_train)
X_test_tfidf = tfidf.transform(narratives_test)

print(f'TF-IDF shape: {X_train_tfidf.shape}')
print(f'Vocabulary size: {len(tfidf.get_feature_names_out())}')

In [ ]:
# Prepare structured features (drop narrative column)
X_train_structured = X_train.drop('narrative_clean', axis=1)
X_test_structured = X_test.drop('narrative_clean', axis=1)

# Standardize numeric features
scaler = StandardScaler()
X_train_structured = scaler.fit_transform(X_train_structured)
X_test_structured = scaler.transform(X_test_structured)

print(f'Structured features shape: {X_train_structured.shape}')

In [ ]:
# Evaluation metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, roc_curve, auc
)

def evaluate_model(y_true, y_pred, y_proba=None, model_name='Model'):
    metrics = {
        'Model': model_name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1': f1_score(y_true, y_pred, zero_division=0),
    }
    if y_proba is not None:
        metrics['ROC-AUC'] = roc_auc_score(y_true, y_proba)
        metrics['PR-AUC'] = average_precision_score(y_true, y_proba)
    return metrics

results = []

## 1. Baseline Models (Text-only)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

# Logistic Regression on TF-IDF
print('Training Logistic Regression...')
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_tfidf, y_train)
y_pred_lr = lr.predict(X_test_tfidf)
y_proba_lr = lr.predict_proba(X_test_tfidf)[:, 1]
results.append(evaluate_model(y_test, y_pred_lr, y_proba_lr, 'Logistic Regression'))

# Naive Bayes on TF-IDF
print('Training Naive Bayes...')
nb = MultinomialNB(alpha=0.1)
nb.fit(X_train_tfidf, y_train)
y_pred_nb = nb.predict(X_test_tfidf)
y_proba_nb = nb.predict_proba(X_test_tfidf)[:, 1]
results.append(evaluate_model(y_test, y_pred_nb, y_proba_nb, 'Naive Bayes'))

print('Baseline models trained.')

## 2. KNN (on TF-IDF with dimensionality reduction)

In [ ]:
from sklearn.decomposition import TruncatedSVD
from sklearn.neighbors import KNeighborsClassifier

# Reduce dimensionality for KNN (dense input faster)
print('Reducing TF-IDF dimensionality for KNN...')
svd = TruncatedSVD(n_components=50, random_state=42)
X_train_svd = svd.fit_transform(X_train_tfidf)
X_test_svd = svd.transform(X_test_tfidf)

print('Training KNN (k=5)...')
knn = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
knn.fit(X_train_svd, y_train)
y_pred_knn = knn.predict(X_test_svd)
y_proba_knn = knn.predict_proba(X_test_svd)[:, 1]
results.append(evaluate_model(y_test, y_pred_knn, y_proba_knn, 'KNN (k=5)'))

print('KNN model trained.')

## 3. Tree Ensemble (structured + text features combined)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Combine structured + text features
print('Combining structured and text features...')
X_train_combined = np.hstack([
    X_train_structured,
    X_train_tfidf.toarray()[:, :100]  # Top 100 TF-IDF features
])
X_test_combined = np.hstack([
    X_test_structured,
    X_test_tfidf.toarray()[:, :100]
])

print('Training Random Forest...')
rf = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train_combined, y_train)
y_pred_rf = rf.predict(X_test_combined)
y_proba_rf = rf.predict_proba(X_test_combined)[:, 1]
results.append(evaluate_model(y_test, y_pred_rf, y_proba_rf, 'Random Forest'))

print('✓ Random Forest trained.')

## 4. Neural Network (ANN)

In [ ]:
from sklearn.neural_network import MLPClassifier

print('Training Neural Network (ANN)...')
ann = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu',
    solver='adam',
    max_iter=500,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    learning_rate_init=0.001
)
ann.fit(X_train_combined, y_train)
y_pred_ann = ann.predict(X_test_combined)
y_proba_ann = ann.predict_proba(X_test_combined)[:, 1]
results.append(evaluate_model(y_test, y_pred_ann, y_proba_ann, 'Neural Network (ANN)'))

print('✓ Neural Network trained.')

## 5. Results & Model Comparison

In [ ]:
# Convert to dataframe and sort by F1 score
results_df = pd.DataFrame(results).sort_values('F1', ascending=False)

print('\n' + '='*90)
print('MODEL COMPARISON RESULTS')
print('='*90)
print(results_df.round(4).to_string())

# Save results
results_df.to_csv('../reports/model_comparison.csv', index=False)
print(f'\n✓ Results saved to reports/model_comparison.csv')

## 6. Voting Ensemble (Soft Voting)

In [ ]:
# Manual soft voting: average probabilities from all models
print('Building soft-vote ensemble (averaging probabilities)...')

# Average predictions from all models
y_proba_ensemble = (
    y_proba_lr +       # Logistic Regression
    y_proba_nb +       # Naive Bayes
    y_proba_knn +      # KNN
    y_proba_rf +       # Random Forest
    y_proba_ann        # Neural Network
) / 5

y_pred_ensemble = (y_proba_ensemble >= 0.5).astype(int)

results.append(evaluate_model(y_test, y_pred_ensemble, y_proba_ensemble, 'Voting Ensemble (5-model avg)'))

print('✓ Voting ensemble created.')

In [ ]:
# Final results with ensemble
final_results = pd.DataFrame(results).sort_values('F1', ascending=False)

print('\n' + '='*90)
print('FINAL RESULTS (INCLUDING ENSEMBLE)')
print('='*90)
print(final_results.round(4).to_string())

# Save final results
final_results.to_csv('../reports/model_comparison_with_ensemble.csv', index=False)
print(f'\n✓ Final results saved to reports/model_comparison_with_ensemble.csv')

## Summary & Next Steps

In [ ]:
best_model = final_results.iloc[0]

print('\n' + '='*90)
print('PROJECT SUMMARY')
print('='*90)
print(f"\nBest Model: {best_model['Model']}")
print(f"  F1 Score: {best_model['F1']:.4f}")
print(f"  Precision: {best_model['Precision']:.4f}")
print(f"  Recall: {best_model['Recall']:.4f}")
if 'ROC-AUC' in best_model and pd.notna(best_model['ROC-AUC']):
    print(f"  ROC-AUC: {best_model['ROC-AUC']:.4f}")
if 'PR-AUC' in best_model and pd.notna(best_model['PR-AUC']):
    print(f"  PR-AUC: {best_model['PR-AUC']:.4f}")

print(f"\nModels Trained: {len(results)}")
print(f"  - 2 baseline (Logistic Regression, Naive Bayes)")
print(f"  - 1 KNN (k=5)")
print(f"  - 1 Random Forest (100 trees)")
print(f"  - 1 Neural Network (3-layer ANN)")
print(f"  - 1 Voting Ensemble (5-model soft average)")

print(f"\n✓ Model comparison complete!")
print(f"✓ Results in: reports/model_comparison_with_ensemble.csv")